In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
secret_value_1 = user_secrets.get_secret("kaggle_key")
secret_value_2 = user_secrets.get_secret("KAGGLE_USERNAME")

In [ ]:
%cd /kaggle/working

!rm -rf /kaggle/working/ComfyUI

!git clone https://github.com/comfyanonymous/ComfyUI.git /kaggle/working/ComfyUI

%cd /kaggle/working/ComfyUI

!python -m pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt

print("ComfyUI installed")

In [ ]:
%cd /kaggle/working/ComfyUI

!test -f manager_requirements.txt && pip install -q -r manager_requirements.txt || echo "manager_requirements.txt not found"

print("Manager dependencies installed")

In [ ]:
%cd /kaggle/working/ComfyUI/custom_nodes

!rm -rf /kaggle/working/ComfyUI/custom_nodes/ComfyUI-GGUF
!git clone https://github.com/city96/ComfyUI-GGUF.git /kaggle/working/ComfyUI/custom_nodes/ComfyUI-GGUF

!pip install -q --upgrade gguf

!test -f /kaggle/working/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt && \
pip install -q -r /kaggle/working/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt || \
echo "No extra GGUF requirements file"

print("GGUF node installed")
print("Put GGUF UNet files in: /kaggle/working/ComfyUI/models/unet")

In [ ]:
%cd /kaggle/working/ComfyUI
!python main.py --listen 0.0.0.0 --port 8188 --enable-manager --lowvram --preview-method none

In [ ]:
!cd /kaggle/working/

In [ ]:
import subprocess

url = "https://github.com/openziti/zrok/releases/download/v2.0.2/zrok_2.0.2_linux_amd64.tar.gz"

print("Downloading:", url)
subprocess.run(["wget", "-O", "zrok2.tar.gz", url], check=True)
subprocess.run(["tar", "-xzf", "zrok2.tar.gz"], check=True)

In [ ]:
!ls -lah /kaggle/working/zrok2

In [ ]:
!chmod +x /kaggle/working/zrok2
!/kaggle/working/zrok2 version

In [ ]:
# SECURITY FIX: Load ZROK_TOKEN from Kaggle Secrets instead of hardcoding
# Add your zrok token to Kaggle Secrets with the key name: ZROK_TOKEN
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
zrok_token = user_secrets.get_secret("ZROK_TOKEN")
import subprocess
subprocess.run(["/kaggle/working/zrok2", "enable", "-v", zrok_token], check=True)

In [ ]:
import subprocess
import threading
import time
import os

!pkill -9 zrok2

zrok_path = "/kaggle/working/zrok2"
comfy_path = "/kaggle/working/ComfyUI"

def start_zrok():
    global public_url
    process = subprocess.Popen(
        [zrok_path, "share", "public", "http://localhost:8188", "--headless"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    
    print("Zrok Log Feed Started:")
    for line in iter(process.stdout.readline, ''):
        clean_line = line.strip()
        if clean_line:
            print(f"  [zrok] {clean_line}")
        
        if "access your share at" in line:
            public_url = line.split("at ")[1].strip()
            print(f"PUBLIC URL FOUND: {public_url}")

public_url = None
thread = threading.Thread(target=start_zrok, daemon=True)
thread.start()

print("Waiting for zrok to handshake...")
for _ in range(30):
    if public_url: break
    time.sleep(1)

if not public_url:
    print("Warning: URL not caught by script yet, but check the logs above!")

print(f"Launching ComfyUI...")
os.chdir(comfy_path)
!python main.py --listen 0.0.0.0 --port 8188 --enable-manager --lowvram --preview-method none